# Chapter 14 &mdash; The Decidability Venn Diagram

**Concept 11 of the Chapter 14 decomposition:** *The Decidability Venn Diagram: Closure and Decidability Summarized*

All the closure results in one picture &mdash; plus the counting proof that non-RE languages exist.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-Decidability-Venn-Diagram/Concept-Decidability-Venn-Diagram.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The whole landscape in one diagram:

* **recursive** $\subsetneq$ **RE** $\subsetneq$ **all languages**;
* recursive languages are closed under **union, intersection, complement,
  concatenation and star**;
* RE languages are closed under **union, intersection, concatenation and star** &mdash;
  but **not complement**;
* $L$ recursive $\iff$ $L$ and $\overline{L}$ both RE.

And the reason the outer containment is strict is a **counting** argument. There are
**countably many** Turing machines (each is a finite string over a finite alphabet) and
**uncountably many** languages (each is a subset of the countable set $\Sigma^*$, and
$2^{\aleph_0} > \aleph_0$).

So almost every language has no machine at all. The named examples like
$\overline{A_{TM}}$ are just the ones we can point at.

## 2. Definitions

### The closure table

In [ ]:
CLOSURE = [
 ("union",         "yes", "yes"),
 ("intersection",  "yes", "yes"),
 ("complement",    "yes", "NO"),
 ("concatenation", "yes", "yes"),
 ("star",          "yes", "yes"),
 ("difference",    "yes", "NO"),
]

### The counting argument, made concrete

In [ ]:
from itertools import product
def machine_descriptions(upto, sigma='01'):
    # every TM is SOME finite string; here we just count the strings
    return sum(len(sigma) ** k for k in range(upto + 1))

def languages_over(upto, sigma='01'):
    # subsets of the strings of length <= upto
    n = sum(len(sigma) ** k for k in range(upto + 1))
    return 2 ** n

<!-- nav-strip -->

---

&larr;&nbsp;[Ch14&nbsp;10.&nbsp;$A_{TM}$, and Why RE Languages Are Called ](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-A-TM-And-Enumerability/Concept-A-TM-And-Enumerability.ipynb) &nbsp;&middot;&nbsp; [**Chapter 14** index](https://github.com/ganeshutah/Jove/blob/master/Chapter14/README.md) &nbsp;&middot;&nbsp; [Ch14&nbsp;12.&nbsp;High-Level Proof Sketches, and the Strictness of the Hierarchy](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter14/Concept-High-Level-Proof-Sketches/Concept-High-Level-Proof-Sketches.ipynb)&nbsp;&rarr;

---

## 3. Tests

The closure table.

In [ ]:
print("%-16s %-12s %s" % ("operation", "recursive", "RE"))
for op, rec, re_ in CLOSURE:
    print("%-16s %-12s %s" % (op, rec, re_))
print()
print("The single NO in the recursive column is... there is none.")
print("The NOs in the RE column are what make RE interesting.")

Why RE is **not** closed under complement.

In [ ]:
print("Suppose RE were closed under complement.")
print("Then for any RE language L, both L and complement(L) would be RE,")
print("so by Concept 7 every RE language would be RECURSIVE.")
print("But A_TM is RE and not recursive.  Contradiction.")

**Counting:** far more languages than machines, at every finite size.

In [ ]:
print("%-8s %-16s %s" % ("length", "descriptions", "languages"))
for n in range(1, 6):
    print("%-8d %-16d %d" % (n, machine_descriptions(n), languages_over(n)))
for n in range(1, 6):
    assert languages_over(n) > machine_descriptions(n)
print("\nAnd the gap is not merely large -- in the limit it is a difference")
print("of CARDINALITY: countably many machines, uncountably many languages.")

Diagonalization, in miniature, to show the gap is real.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(4) for p in product('01', repeat=k)]
# pretend these five 'machines' are all we have
fakes = [lambda w: w.startswith('1'),
         lambda w: w.count('0') % 2 == 0,
         lambda w: len(w) > 2,
         lambda w: w == '',
         lambda w: '01' in w]
diag = {w for i, w in enumerate(strs[:len(fakes)]) if not fakes[i](w)}
print("the diagonal language starts :", sorted(diag))
for i, f in enumerate(fakes):
    w = strs[i]
    assert (w in diag) != f(w)
print("differs from every listed machine at its own diagonal string --")
print("so it is recognised by none of them.  Now do that for ALL machines.")

The picture, in words.

In [ ]:
print("+-------------------------------------------------+")
print("|  all languages                                  |")
print("|   +-----------------------------------------+   |")
print("|   |  RE (recognizable)      A_TM            |   |")
print("|   |   +---------------------------------+   |   |")
print("|   |   |  recursive (decidable)          |   |   |")
print("|   |   |     L_EmptyDFA, every regular   |   |   |")
print("|   |   +---------------------------------+   |   |")
print("|   +-----------------------------------------+   |")
print("|      complement(A_TM) lives out here            |")
print("+-------------------------------------------------+")

## 4. Exercises


1. Prove RE is closed under union by constructing the machine.
2. Where does that construction break for complement?
3. Give a language that is neither RE nor co-RE.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 246 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter14/Concept-Decidability-Venn-Diagram')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')